In [1]:
import numpy as np
import cv2
import librosa
import soundfile as sf

# ============ NEW (Week 1): frequency remapping ============
def frequency_remap(spectrogram, sr=22050, n_fft=1024, target_low=1000.0, target_high=4000.0):
    """
    Warps a linear-frequency magnitude spectrogram so ALL of its structural
    content is compressed into the [target_low, target_high] Hz band.

    f_new = a * log(f_old + 1) + b   maps [0, sr/2] -> [target_low, target_high]

    We need the INVERSE (a "pull"/backward mapping): for every output bin
    whose true physical frequency falls in the target band, ask "which
    original frequency warped TO here?" and interpolate that value in.
    Bins outside the band are left at zero (silence).
    """
    n_bins, n_frames = spectrogram.shape
    freq_axis = np.linspace(0, sr / 2, n_bins)  # true Hz of each bin, 0..11025

    b = target_low
    a = (target_high - target_low) / np.log(freq_axis[-1] + 1)

    remapped = np.zeros_like(spectrogram)
    in_band = (freq_axis >= target_low) & (freq_axis <= target_high)
    f_true = freq_axis[in_band]

    f_source = np.exp((f_true - b) / a) - 1.0
    src_bin_pos = np.clip(f_source / (sr / n_fft), 0, n_bins - 1)

    src_bins = np.arange(n_bins)
    for t in range(n_frames):
        remapped[in_band, t] = np.interp(src_bin_pos, src_bins, spectrogram[:, t])

    return remapped, a, b
# ============ END NEW ============


def image_to_spectrogram_audio(image_path, output_audio="output.wav", sr=22050, apply_remap=True):  # NEW: apply_remap flag added
    # Load image in grayscale
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError("Image not found or invalid path.")
    
    # Resize image to match Griffin-Lim constraints (n_fft = 1024, hop_length=512)
    # Height (frequency bins) must be (n_fft // 2 + 1) = 513 for n_fft=1024
    target_height = 513  # For n_fft=1024
    target_width = 128   # Adjust width as needed (e.g., 128 for time steps)
    image = cv2.resize(image, (target_width, target_height))
    
    # Normalize to [0, 1] and flip vertically (spectrograms use bottom-to-top frequency)
    image_normalized = np.flipud(image.astype(np.float32) / 255.0)
    
    # Convert image to dB-scaled "spectrogram" (add epsilon to avoid log(0))
    spectrogram_db = librosa.amplitude_to_db(image_normalized + 1e-7, ref=np.max)
    
    # Convert dB back to linear amplitude
    spectrogram_linear = librosa.db_to_amplitude(spectrogram_db)
    
    # Griffin-Lim parameters (must match spectrogram dimensions)
    n_fft = 1024
    hop_length = 512
    win_length = 1024
    
    # ============ NEW (Week 1): apply frequency remap here ============
    if apply_remap:
        spectrogram_linear, a, b = frequency_remap(spectrogram_linear, sr=sr, n_fft=n_fft)
        print(f"  remap params -> a={a:.2f}, b={b:.2f}")
    # ============ END NEW ============
    
    # Reconstruct audio
    audio = librosa.griffinlim(
        spectrogram_linear,
        n_iter=32,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length
    )
    
    # Save as WAV file
    sf.write(output_audio, audio, sr)
    print(f"Audio saved to {output_audio}")

# Example usage — generate both to compare
# image_to_spectrogram_audio(r"D:\Documents\Iquisitionis\103\tb_plots\car_test.jp", "car_original.wav", apply_remap=False)
# image_to_spectrogram_audio(r"D:\Documents\Iquisitionis\103\vehicle-type-recognition\Dataset\Car\Image_32.jpg", "car_remapped2.wav", apply_remap=True)

In [2]:
# def band_energy_fraction(audio_path, sr=22050, n_fft=2048, hop_length=512, band=(1000, 4000)):
#     """What fraction of the audio's REAL energy lands in the target band.
#     Uses n_fft=2048 to match your paper's Section 3.3 re-analysis settings."""
#     y, _ = librosa.load(audio_path, sr=sr)
#     S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length, window='hann')) ** 2
#     freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
#     power_per_bin = S.mean(axis=1)
#     in_band = (freqs >= band[0]) & (freqs <= band[1])
#     return power_per_bin[in_band].sum() / power_per_bin.sum()

# # print("Original :", band_energy_fraction("car_original.wav"))
# print("Remapped :", band_energy_fraction("car_remapped.wav"))

In [3]:
import os

# ============ NEW: batch/dataset conversion, preserving folder structure ============
def process_dataset(input_path, output_path, apply_remap=True, sr=22050):
    """
    Converts every image under `input_path` into a .wav file.

    - `input_path` is a single image file -> converts just that file
      (identical behavior to calling image_to_spectrogram_audio directly).
    - `input_path` is a dataset folder    -> walks it recursively and
      recreates the EXACT same subfolder structure under a new output
      folder, so the original dataset is never touched or overwritten.

    Only `input_path` is required. `output_path` is optional and defaults
    to "<input_path>_wav", created as a sibling of the input (never nested
    inside it, so the walk never picks up its own output as it runs).
    """
    IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"No such file or directory: {input_path}")

    # ---- single file: identical behavior to calling image_to_spectrogram_audio directly ----
    if os.path.isfile(input_path):
        out_file = output_path or (os.path.splitext(input_path)[0] + ".wav")
        image_to_spectrogram_audio(input_path, out_file, sr=sr, apply_remap=apply_remap)
        return

    # ---- dataset folder: mirror the exact subfolder structure ----
    input_path  = os.path.normpath(input_path)
    output_root = output_path + "v2" #or (input_path + "_wav")

    n_done, n_skipped = 0, 0
    for root, _dirs, files in os.walk(input_path):
        rel_dir = os.path.relpath(root, input_path)
        out_dir = output_root if rel_dir == "." else os.path.join(output_root, rel_dir)
        os.makedirs(out_dir, exist_ok=True)

        for fname in files:
            if os.path.splitext(fname)[1].lower() not in IMG_EXTS:
                continue  # non-image files (csv, txt, etc.) are silently ignored
            in_file  = os.path.join(root, fname)
            out_file = os.path.join(out_dir, os.path.splitext(fname)[0] + ".wav")
            try:
                image_to_spectrogram_audio(in_file, out_file, sr=sr, apply_remap=apply_remap)
                n_done += 1
            except Exception as e:
                n_skipped += 1  # bad/corrupt file -> skip it, don't kill the whole run
                print(f"  [skip] {in_file}: {e}")

    print(f"\nDone -> {output_root}  (converted: {n_done}, skipped: {n_skipped})")
# ============ END NEW ============

# Example usage - one argument works for either a single image or a whole dataset folder
process_dataset(r"D:\Documents\Iquisitionis\103\vehicle-type-recognition\Dataset", r"D:\Documents\Iquisitionis\103v2\data\4Class_remapped")


  remap params -> a=322.30, b=1000.00
Audio saved to D:\Documents\Iquisitionis\103v2\data\4Class_remappedv2\Bus\Image_1.wav
  remap params -> a=322.30, b=1000.00
Audio saved to D:\Documents\Iquisitionis\103v2\data\4Class_remappedv2\Bus\Image_10.wav
  remap params -> a=322.30, b=1000.00
Audio saved to D:\Documents\Iquisitionis\103v2\data\4Class_remappedv2\Bus\Image_100.wav
  remap params -> a=322.30, b=1000.00
Audio saved to D:\Documents\Iquisitionis\103v2\data\4Class_remappedv2\Bus\Image_11.wav
  remap params -> a=322.30, b=1000.00
Audio saved to D:\Documents\Iquisitionis\103v2\data\4Class_remappedv2\Bus\Image_12.wav
  remap params -> a=322.30, b=1000.00
Audio saved to D:\Documents\Iquisitionis\103v2\data\4Class_remappedv2\Bus\Image_13.wav
  remap params -> a=322.30, b=1000.00
Audio saved to D:\Documents\Iquisitionis\103v2\data\4Class_remappedv2\Bus\Image_14.wav
  remap params -> a=322.30, b=1000.00
Audio saved to D:\Documents\Iquisitionis\103v2\data\4Class_remappedv2\Bus\Image_15.wav
